In [ ]:
import numpy as np
seed=42
# Method 1
np.random.seed(seed)
rand_vals_method1 = np.random.rand(5)
print(rand_vals_method1)

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]


In [ ]:
# Method 2
bit_generator = np.random.MT19937(seed)
rng = np.random.default_rng(bit_generator)
rand_vals_method2 = rng.random(5)
print(rand_vals_method2)


[0.54199389 0.61966721 0.05736978 0.81190365 0.86009402]


In [ ]:
# Method 3
bit_generator2 = np.random.MT19937(seed)
rng2 = np.random.Generator(bit_generator2)

rand_vals_method3 = rng2.random(5)
print(rand_vals_method3)


[0.54199389 0.61966721 0.05736978 0.81190365 0.86009402]


In [ ]:
# Method 4
bit_generator3 = np.random.MT19937()
bit_generator3._legacy_seeding(seed)
rng3 = np.random.Generator(bit_generator3)

rand_vals_method4 = rng3.random(5)
print(rand_vals_method4)

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]


In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)
print(rng)
arr1 = rng.random((3, 3))
arr1

Generator(PCG64)


array([[0.77395605, 0.43887844, 0.85859792],
       [0.69736803, 0.09417735, 0.97562235],
       [0.7611397 , 0.78606431, 0.12811363]])

In [ ]:
#philox4x64 10 0000000000000000 0000000000000000 0000000000000000 0000000000000000 0000000000000000 0000000000000000   16554d9eca36314c db20fe9d672d0fdc d7e772cee186176b 7e68b68aec7ba23b
#philox4x64 10 ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff   87b092c3013fe90b 438c3c67be8d0224 9cc7d7c69cd777b6 a09caebf594f0ba0
#philox4x64 10 243f6a8885a308d3 13198a2e03707344 a4093822299f31d0 082efa98ec4e6c89 452821e638d01377 be5466cf34e90c6c   a528f45403e61d95 38c72dbd566e9788 a5a1610e72fd18b5 57bd43b5e52b7fe6



In [ ]:

#Using numpy without any prior correction
import numpy as np
from randomgen import Philox
Counter = (0,0,0,0)
key = (0,0)
bit_generator = Philox(counter = Counter, key = key)
# The random_raw method is called directly on the Philox bit generator
uniform_values = bit_generator.random_raw(size=4) # specify size for multiple values
hex_digits = [format(x, '016x') for x in uniform_values]
print(hex_digits)

['02f4ba6408e4d89b', '3dd62b0b9ca8c5b2', '1c8667a55d902e79', '907d7a052fd5b4dc']


In [ ]:
from numpy.random import Generator, Philox
rg = Generator(Philox(1234))
rg.standard_normal()

-0.7570164779736382

In [ ]:
N = 4
# Quelle valeur de Seed? car il y a une transformation intermédiaire  afin d'accéder aux paramètres
rng = np.random.Generator(np.random.Philox(123))
xs = rng.integers(0, 2**64, size=N, dtype=np.uint64)

hex_digits = [format(x, '016x') for x in xs]
print(hex_digits)


['e66b69f742f0ad65', 'e7ee8ac77da40e6a', '3f31a3dff895b9d7', 'c279d8b8d3cc316d']


In [ ]:

# En utilisant le counter et les keys directement pour contournant l'utilisation de la seed

import numpy as np
#counter = [0x00, 0x00, 0x00, 0x00]
counter = [0000000000000000, 0000000000000000, 0000000000000000, 0000000000000000]
#key = [0x00,0x00]
key = [0000000000000000, 0000000000000000]
bit_generator = np.random.Philox(counter = counter, key = key )
rng = np.random.Generator(bit_generator)

X = rng.integers(
    low=0,
    high=2**64, size = 4,
    dtype=np.uint64
)
hex_values = [format(x, '016x') for x in X]
print(hex_values)


['02f4ba6408e4d89b', '3dd62b0b9ca8c5b2', '1c8667a55d902e79', '907d7a052fd5b4dc']


philox4x64 10 0000000000000000 0000000000000000 0000000000000000 0000000000000000 0000000000000000 0000000000000000   16554d9eca36314c db20fe9d672d0fdc d7e772cee186176b 7e68b68aec7ba23b

### on peut remarquer que les sorties sont différentes maintenant nous allons implémenter from scratch random123 en python à fin de pouvoir mieux analyser les sorties.

In [ ]:
MASK = 0xFFFFFFFFFFFFFFFF

M0 = 0xD2B74407B1CE6E93
M1 = 0xCA5A826395121157
WEYL = 0x9E3779B97F4A7C15

def mul64(a, b):
    p = a * b
    return (p >> 64) & MASK, p & MASK

def philox4x64(counter, key, rounds=10):
    c0, c1, c2, c3 = counter
    k0, k1 = key

    for _ in range(rounds):
        hi0, lo0 = mul64(c0, M0)
        hi1, lo1 = mul64(c2, M1)

        c0 = hi1 ^ c1 ^ k0
        c1 = lo1
        c2 = hi0 ^ c3 ^ k1
        c3 = lo0

        k0 = (k0 + WEYL) & MASK
        k1 = (k1 + WEYL) & MASK

    return c0, c1, c2, c3


In [ ]:
out = philox4x64(
    counter=(0,0,0,0),
    key=(0,0)
)

print([format(x, '016x') for x in out])


['c4a2b21aa3d49181', 'a0a4bf7662be31e5', 'cf4aa071a58e7d5d', 'c9ac9d4ff4e5437a']


Même avec une implémentation simple, on peut rémarquer que les séquences sont toujours différentes.

Et pourquoi?


In [ ]:
#En utilisant randomgen, utilisation de d'autres bibliothèques afin d'investiguer encore plus si c'est un problème de code ou de bibliothèques

!pip install randomgen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 17.6 MB/s eta 0:00:00


In [ ]:
#Génération avec counter et key
bg = Philox(counter=[0,0,0,0], key=[0,0], number=4, width=64)

# sortie brute Philox (uint64)
out = bg.random_raw(4)
print([hex(x) for x in out])


['0x2f4ba6408e4d89b', '0x3dd62b0b9ca8c5b2', '0x1c8667a55d902e79', '0x907d7a052fd5b4dc']


On peut rémarquer que la sortie est toujours différentes à celle de la référence mais elle reste identique à celle de numpy ce qui ne résout pas notre problème de portabilité donc quel est le problème?

In [ ]:
from randomgen import Philox
from numpy.random import Philox as np_philox
from numpy.random import Generator
counter = (0,0,0,0)
key = (0,0)
rng = Generator(np_philox(counter= counter, key = key))
X = rng.integers(0, 2**64, size = 4, dtype='uint64')
hex_values = [format(x, '016x') for x in X]
print(hex_values)
state = rng.bit_generator.state
print(state)

['02f4ba6408e4d89b', '3dd62b0b9ca8c5b2', '1c8667a55d902e79', '907d7a052fd5b4dc']
{'bit_generator': 'Philox', 'state': {'counter': array([1, 0, 0, 0], dtype=uint64), 'key': array([0, 0], dtype=uint64)}, 'buffer': array([  213000021201967259,  4455796210202625458,  2055444239878205049,
       10411612076246414556], dtype=uint64), 'buffer_pos': 4, 'has_uint32': 0, 'uinteger': 0}


Ici on a utilisé bit_generator.state afin de voir un peu ce qui se passe à l'intérieur de philox pour la bibliothèque numpy et nous allons découvrir qu'au lieu d'avoir:
Counter = (0,0,0,0) comme en entrée on a un changement et Counter = (1,0,0,0)  Ce qui va vraiment nous donner un coups de pouce dans notre récherche.

In [ ]:
MASK64 = 0xFFFFFFFFFFFFFFFF

PHILOX_M4_64_0 = 0xD2E7470EE14C6C93
PHILOX_M4_64_1 = 0xCA5A826395121157
PHILOX_W_64_0  = 0x9E3779B97F4A7C15
PHILOX_W_64_1  = 0xBB67AE8584CAA73B



def mulhilo(a, b):
    p = a * b
    return (p >> 64) & MASK64, p & MASK64


def philox4x64_round(c, k):
    c0, c1, c2, c3 = c
    k0, k1 = k

    hi0, lo0 = mulhilo(PHILOX_M4_64_0, c0)
    hi1, lo1 = mulhilo(PHILOX_M4_64_1, c2)

    return (
        hi1 ^ c1 ^ k0,
        lo1,
        hi0 ^ c3 ^ k1,
        lo0,
    )


def bumpkey(k):
    return (
        (k[0] + PHILOX_W_64_0) & MASK64,
        (k[1] + PHILOX_W_64_1) & MASK64,
    )


def philox4x64(counter, key, rounds=10):
    c = counter
    k = key
    for _ in range(rounds):
        c = philox4x64_round(c, k)
        k = bumpkey(k)
    return c


In [ ]:
counter1 = (0, 0, 0, 0)
key1 = (0, 0)
out1 = philox4x64(counter1, key1)

print([hex(x) for x in out1])

['0x16554d9eca36314c', '0xdb20fe9d672d0fdc', '0xd7e772cee186176b', '0x7e68b68aec7ba23b']


On a réimplémenté philox en python en utilisant le code source fait en langage C et malgré que l'implémentation reste la même que celle qu'on avait.
Cette fois-ci on a pris les constantes dans le code source en C (code originale).

Nous allons retenir que les constantes sont les clés pour une bonne implémentation de philox.

In [ ]:
# Random123 avec counter=N  →  NumPy avec counter=N+1

import numpy as np
from numpy.random import Philox

bit_gen = Philox()
state = bit_gen.state
state['state']['counter'] = np.array([0, 0, 0, 0], dtype=np.uint64)  # +1 !
state['state']['key'] = np.array([0, 0], dtype=np.uint64)
state['state']['buffer_pos'] = 4
bit_gen.state = state

# Maintenant NumPy génère les mêmes valeurs que Random123 avec counter={0}

rng = np.random.Generator(bit_gen)
X = rng.integers(0, 2**64, size = 4, dtype='uint64')
hex_values = [format(x, '016x') for x in X]
print(hex_values)

['02f4ba6408e4d89b', '3dd62b0b9ca8c5b2', '1c8667a55d902e79', '907d7a052fd5b4dc']


Nous avons maintenant Compris qu'en prenant  Counter = N  en entrée et après génération Counter = N+1 (Numpy).

Cela va nous permettre de démontrer la portabilité.

In [ ]:
import numpy as np
from numpy.random import Philox

# Initialisation pour obtenir la séquence Random123 depuis counter={0,0,0,0}
bit_gen = Philox()
state = bit_gen.state

# Le truc: 0xFFFFFFFFFFFFFFFF est -1 en uint64
# Après incrémentation de NumPy: 0xFFFFFFFFFFFFFFFF + 1 = 0x0 (wraparound)
state['state']['counter'] = np.array([
    0xFFFFFFFFFFFFFFFF,  # C'est -1 !
    0xFFFFFFFFFFFFFFFF,
    0xFFFFFFFFFFFFFFFF,
    0xFFFFFFFFFFFFFFFF
], dtype=np.uint64)

state['state']['key'] = np.array([0,0], dtype=np.uint64)
state['state']['buffer_pos'] = 4
bit_gen.state = state

# Générer les valeurs - identiques à Random123 avec counter={0,0,0,0}
values = [bit_gen.random_raw(1)[0] for _ in range(4)]

print("Valeurs générées (hex):")
for i, v in enumerate(values):
    print(f"  [{i}]: {v:016x}")

Valeurs générées (hex):
  [0]: 16554d9eca36314c
  [1]: db20fe9d672d0fdc
  [2]: d7e772cee186176b
  [3]: 7e68b68aec7ba23b


Et maintenant, Nous avons pu obtenir le résultat Counter(Numpy) = Counter(random) + 1.

In [ ]:
#philox4x64 10 ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff ffffffffffffffff   87b092c3013fe90b 438c3c67be8d0224 9cc7d7c69cd777b6 a09caebf594f0ba0

import numpy as np
from numpy.random import Philox

# Initialisation
bit_gen = Philox()
state = bit_gen.state

# CORRECT: Décrémenter seulement counter[0]
state['state']['counter'] = np.array([
    0xFFFFFFFFFFFFFFFE,  # counter[0] - 1
    0xFFFFFFFFFFFFFFFF,  # counter[1] inchangé
    0xFFFFFFFFFFFFFFFF,  # counter[2] inchangé
    0xFFFFFFFFFFFFFFFF   # counter[3] inchangé
], dtype=np.uint64)

state['state']['key'] = np.array([
    0xFFFFFFFFFFFFFFFF,
    0xFFFFFFFFFFFFFFFF
], dtype=np.uint64)

state['state']['buffer_pos'] = 4
bit_gen.state = state

# Générer les valeurs
values = [bit_gen.random_raw(1)[0] for _ in range(4)]

print("Valeurs générées (hex):")
for i, v in enumerate(values):
    print(f"  [{i}]: {v:016x}")

print("\nValeurs Random123 attendues:")
print("  [0]: 87b092c3013fe90b")
print("  [1]: 438c3c67be8d0224")
print("  [2]: 9cc7d7c69cd777b6")
print("  [3]: a09caebf594f0ba0")

Valeurs générées (hex):
  [0]: 87b092c3013fe90b
  [1]: 438c3c67be8d0224
  [2]: 9cc7d7c69cd777b6
  [3]: a09caebf594f0ba0

Valeurs Random123 attendues:
  [0]: 87b092c3013fe90b
  [1]: 438c3c67be8d0224
  [2]: 9cc7d7c69cd777b6
  [3]: a09caebf594f0ba0


# Après correction: Perfect result

In [ ]:
import numpy as np
from numpy.random import Philox

# Valeurs Random123 de référence
# counter = [0x243f6a8885a308d3, 0x13198a2e03707344, 0xa4093822299f31d0, 0x082efa98ec4e6c89]
# key = [0x452821e638d01377, 0xbe5466cf34e90c6c]

# Pour NumPy : counter - 1
counter_numpy = [
    0x243f6a8885a308d2,  # 0x243f6a8885a308d3 - 1
    0x13198a2e03707344,  # inchangé
    0xa4093822299f31d0,  # inchangé
    0x082efa98ec4e6c89   # inchangé
]

key = [0x452821e638d01377, 0xbe5466cf34e90c6c]

# Initialisation
bit_gen = Philox()
state = bit_gen.state

# Modifier l'état
state['state']['counter'] = np.array(counter_numpy, dtype=np.uint64)
state['state']['key'] = np.array(key, dtype=np.uint64)
state['state']['buffer_pos'] = 4  # Buffer vide
bit_gen.state = state

# Générer les valeurs
values = [bit_gen.random_raw(1)[0] for _ in range(4)]

print("Valeurs générées (hex):")
for i, v in enumerate(values):
    print(f"  [{i}]: {v:016x}")

print("\nValeurs Random123 attendues:")
print("  [0]: a528f45403e61d95")
print("  [1]: 38c72dbd566e9788")
print("  [2]: a5a1610e72fd18b5")
print("  [3]: 57bd43b5e52b7fe6")

Valeurs générées (hex):
  [0]: a528f45403e61d95
  [1]: 38c72dbd566e9788
  [2]: a5a1610e72fd18b5
  [3]: 57bd43b5e52b7fe6

Valeurs Random123 attendues:
  [0]: a528f45403e61d95
  [1]: 38c72dbd566e9788
  [2]: a5a1610e72fd18b5
  [3]: 57bd43b5e52b7fe6


# Pytorch

In [ ]:
#philox4x32 10 00000000 00000000 00000000 00000000 00000000 00000000   6627e8d5 e169c58d bc57ac4c 9b00dbd8
#philox4x32 10 ffffffff ffffffff ffffffff ffffffff ffffffff ffffffff   408f276d 41c83b0e a20bc7c6 6d5451fd
#philox4x32 10 243f6a88 85a308d3 13198a2e 03707344 a4093822 299f31d0   d16cfe09 94fdcceb 5001e420 24126ea1

import torch

device = "cuda"

g = torch.Generator(device=device)
g.manual_seed(0)

# générer des uint32
x = torch.randint(
    low=0,
    high=2**32,
    size=(4,),
    dtype=torch.int64,  # PyTorch n'a pas uint32 direct
    generator=g,
    device=device
)

# convertir en uint32 puis hex
hex_values = [format(v.item() & 0xFFFFFFFF, '08x') for v in x]

print(hex_values)

['e169c58d', 'f08d6eaa', 'b6826759', '115896b8']


On peut observer l'apparition de 'e169c58d' et qui correspond à la deuxième valeur philox4x32 et un detail saute aux yeux: Pourquoi une seule valeur?

In [ ]:
#philox4x32 10 00000000 00000000 00000000 00000000 00000000 00000000   6627e8d5 e169c58d bc57ac4c 9b00dbd8
#philox4x32 10 ffffffff ffffffff ffffffff ffffffff ffffffff ffffffff   408f276d 41c83b0e a20bc7c6 6d5451fd

In [ ]:
# Philox4x32-10 (Random123)
# Voici notre implémentation basique de philox4x32 en utilisant la logique de Random123
MASK32 = 0xFFFFFFFF

# Constantes Random123
PHILOX_M4x32_0 = 0xD2511F53
PHILOX_M4x32_1 = 0xCD9E8D57
PHILOX_W32_0   = 0x9E3779B9
PHILOX_W32_1   = 0xBB67AE85


def mulhilo32(a, b):
    product = (a * b) & 0xFFFFFFFFFFFFFFFF
    lo = product & MASK32
    hi = (product >> 32) & MASK32
    return lo, hi


def single_round(counter, key):
    c0, c1, c2, c3 = counter
    k0, k1 = key

    lo0, hi0 = mulhilo32(PHILOX_M4x32_0, c0)
    lo1, hi1 = mulhilo32(PHILOX_M4x32_1, c2)

    return (
        (hi1 ^ c1 ^ k0) & MASK32,
        lo1 & MASK32,
        (hi0 ^ c3 ^ k1) & MASK32,
        lo0 & MASK32
    )


def philox4x32(counter, key, rounds=10):
    c = counter
    k = key

    for _ in range(rounds - 1):
        c = single_round(c, k)
        k = ((k[0] + PHILOX_W32_0) & MASK32,
             (k[1] + PHILOX_W32_1) & MASK32)

    # dernière ronde sans bumpkey
    return single_round(c, k)


In [ ]:
counter1 = (0,0,0,0)

key1 = (0,0)




counter2 = (
    0xffffffff,
    0xffffffff,
    0xffffffff,
    0xffffffff,
)

key2 = (
    0xffffffff,
    0xffffffff,
)


out1 = philox4x32(counter1, key1)

print([hex(x) for x in out1])


out2 = philox4x32(counter2, key2)

#print([hex(x) for x in out2])


['0x6627e8d5', '0xe169c58d', '0xbc57ac4c', '0x9b00dbd8']


Avec la lecture de la documentation : https://blog.codingconfessions.com/p/how-pytorch-generates-random-numbers on a découvre que counter = (offsetlow,offsethigh,subsequencelow, subsequencehigh )
Donc on découvre que pour chaque appel:

Thread 0: subsequence=0, offset=0 → counter = [0, 0, 0, 0]

Thread 1: subsequence=1, offset=0 → counter = [0, 0, 1, 0]

Thread 2: subsequence=2, offset=0 → counter = [0, 0, 2, 0]

...

Thread 1023: subsequence=1023, offset=0 → counter = [0, 0, 1023, 0]

In [ ]:
# Avec Pytorch
#['e169c58d', 'f08d6eaa', 'b6826759', '115896b8', '8f1a3087', 'aa2cfcda', '93eb3d81', '965c2541', 'e47284d4', 'a8326e71', 'd99cfef1', '43ae16e8', '09968978', 'e36db534', '5ececfc1', '6307f2c6', '8ae6e98a', '4d352ee4', '40179967', '34e2d982']

# En variant Counter
#['0x6627e8d5', '0xe169c58d', '0xbc57ac4c', '0x9b00dbd8'] counter = (0,0,0,0)
#['0x844515e1', '0xf08d6eaa', '0xf19c053', '0x83f875f0'] counter = (0,0,1,0)
#['0x661d677', '0xb6826759', '0x262148c6', '0x773c5ff0'] counter = (0,0,2,0)
#['0xf0a90abc', '0x115896b8', '0xbedefe84', '0xb86f45c6'] counter = (0,0,3,0)
#['0xf2237f09', '0x8f1a3087', '0x6397290', '0x13ce167a'] counter = (0,0,4,0)
#['0xcbf6940c', '0xaa2cfcda', '0x7be30b45', '0x1b354e21']  counter = (0,0,5,0)
#['0x6a3e5da7', '0x93eb3d81', '0xcc3925fb', '0xfd14dcbf'] counter = (0,0,6,0)
#['0xd1fc5bdd', '0x965c2541', '0x2b5fd1f3', '0x31bc68b1'] counter = (0,0,7,0)
#['0x3aa29652', '0xe47284d4', '0x111c00e3', '0xeaf508da'] counter = (0,0,8,0)
# ['0xe8daba39', '0xa8326e71', '0x2a56403a', '0xeaab9ebd'] counter = (0,0,9,0)
#['0x1e4b501e', '0xd99cfef1', '0x9df9894', '0x44cb0ac4'] counter = (0,0,10,0)
#['0x1341c655', '0x43ae16e8', '0x7ef483ff', '0x24f74981'] counter = (0,0,11,0)
#['0x68c2b50e', '0x9968978', '0xc8d6e791', '0xa13d6ddc'] counter = (0,0,12,0)

# ['0xf8e4cca4', '0x5cb200db', '0xb1a574eb', '0x97eff67'] counter = (1,0,0,0)
# ['0xca7cf69e', '0xd9c77ded', '0x7e66d954', '0x6d658f40'] counter = (1,0,1,0)
# ['0x78130eff', '0x2299a390', '0xd8704c11', '0x6cdcedc3'] counter = (1,0,2,0)

# En variant les sous-séquences,  on remarque que la colonne portant l'index1 correspond à la séquence générée par pytorch


Maintenant, on va tester deux appels succéssifs de pytorch avec une même initialisation:   


In [ ]:
import torch

g = torch.Generator(device="cuda")
g.manual_seed(0)

x = torch.randint(
    0, 2**32,
    (10,),
    generator=g,
    device="cuda",
    dtype=torch.int64
)
x1 = torch.randint(
    0, 2**32,
    (10,),
    generator=g,
    device="cuda",
    dtype=torch.int64
)

hex_values1 = [format(v.item() & 0xFFFFFFFF, "08x") for v in x1]

hex_values = [format(v.item() & 0xFFFFFFFF, "08x") for v in x]

print(hex_values)
print(hex_values1)

['e169c58d', 'f08d6eaa', 'b6826759', '115896b8', '8f1a3087', 'aa2cfcda', '93eb3d81', '965c2541', 'e47284d4', 'a8326e71']
['5cb200db', 'd9c77ded', '2299a390', 'eaf7db94', '4f101028', '1b5d062e', '78c33ce6', 'c36ac791', '6fd8c369', '1b110732']


In [ ]:
# Philox4x32-10 (Random123)
# Voici notre implémentation basique de philox4x32 en utilisant la logique de Random123
MASK32 = 0xFFFFFFFF

# Constantes Random123
PHILOX_M4x32_0 = 0xD2511F53
PHILOX_M4x32_1 = 0xCD9E8D57
PHILOX_W32_0   = 0x9E3779B9
PHILOX_W32_1   = 0xBB67AE85


def mulhilo32(a, b):
    product = (a * b) & 0xFFFFFFFFFFFFFFFF
    lo = product & MASK32
    hi = (product >> 32) & MASK32
    return lo, hi


def single_round(counter, key):
    c0, c1, c2, c3 = counter
    k0, k1 = key

    lo0, hi0 = mulhilo32(PHILOX_M4x32_0, c0)
    lo1, hi1 = mulhilo32(PHILOX_M4x32_1, c2)

    return (
        (hi1 ^ c1 ^ k0) & MASK32,
        lo1 & MASK32,
        (hi0 ^ c3 ^ k1) & MASK32,
        lo0 & MASK32
    )


def philox4x32(counter, key, rounds=10):
    c = counter
    k = key

    for _ in range(rounds - 1):
        c = single_round(c, k)
        k = ((k[0] + PHILOX_W32_0) & MASK32,
             (k[1] + PHILOX_W32_1) & MASK32)

    # dernière ronde sans bumpkey
    return single_round(c, k)


In [ ]:
counter1 = (1,0,0,0)

key1 = (0,0)




counter2 = (
    1,0,1,0
)

key2 = (
    0,
    0,
)


out1 = philox4x32(counter1, key1)

print([hex(x) for x in out1])


out2 = philox4x32(counter2, key2)

print([hex(x) for x in out2])


['0xf8e4cca4', '0x5cb200db', '0xb1a574eb', '0x97eff67']
['0xca7cf69e', '0xd9c77ded', '0x7e66d954', '0x6d658f40']


Nous avons le pattern vérifié ici aussi.

Donc nous avons la portabilité vérifiée partiellement.